# 07f · 特征表和第一个基线模型

## 这一章在整个项目里的位置

```
00–06 ✅ → [07 特征和基线] → 08 深度学习 → 09 全蛋白组预测 → 10 对外 API
              │
              ├─ 07a 设计 ✅   07b 新数据 ✅   07c 标签 ✅   07d 结构 ✅   07e 语言模型 ✅
              └─ 07f 特征表 + 第一个基线模型 + MLflow   ← 你在这里
```

## 背景：为什么需要这一章

前面五章攒齐了三样东西，但它们还散在三个地方：

| 东西 | 在哪张表 | 哪一章做的 |
|---|---|---|
| 答案（标签） | `gold.fct_variant_label`，107 万个突变 | 07c |
| 结构特征 | `silver.residue_structure`，每个残基一行 | 07d |
| 语言模型特征 | `silver.residue_language_model`，每个残基一行 | 07e |

模型要的是**一张表**：每个突变一行，前面是特征，最后是答案。这一章做三件事：
1. 把它们拼成这张表（**特征表**）
2. 在上面训练**第一个模型**，按 07a 定的规矩考试
3. 把每一次训练**记录在案**，以后任何一个数字都能查到是怎么来的

第一个模型叫**基线**（baseline）：用最朴素的办法先得到一个分数。它本身不是目的，它是 08 号深度学习要跨过的那根**横杆**。

## 你在哪里操作

```
你：在这个 notebook 里运行一格
   ├─ 第 2、3 步：dbt 在 Databricks 上建特征表、跑检查（约 2 分钟）
   ├─ 第 5 步：部署并运行新流水线 baseline：在 Databricks 上训练，结果记进 MLflow（约 13 分钟）
   └─ 第 6、7 步：在这里把结果读回来看
看结果：这里的输出；Databricks 网页的 Catalog、Jobs，还有一个新地方：Experiments
```

## 这一章做的 7 件事

| # | 做什么 | 比喻 |
|---|---|---|
| 1 | 特征表是什么 | 把每个人的体检报告汇成一张总表 |
| 2 | 建表 | 按项目表逐项抄录 |
| 3 | 检查特征 | 抽查有没有漏填、有没有把诊断结果写进了体检项目 |
| 4 | 基线模型和考法 | 对照组 |
| 5 | MLflow：把每次训练记下来 | 实验记录本 |
| 6 | 看结果：过线了吗，每样东西贡献多少 | 敲除实验 |
| 7 | 和旧论文同题对照 | 同一张卷子，两个人答 |

**花费：0**。**时间**：约 20 分钟，其中 15 分钟是等 Databricks。

---

## 准备

运行下面这一格：连上 Databricks，载入这一章的代码。**不用看代码。**

In [ ]:
import os, subprocess, warnings
from pathlib import Path
os.environ.update(MLFLOW_DISABLE_AGENT_HINT="1", MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR="false")   # 少打些无关的提示
warnings.filterwarnings("ignore", message="IProgress not found")
import mlflow
import pandas as pd
from databricks.sdk import WorkspaceClient
from qafi_platform import lakehouse
from qafi_platform.lakehouse import sql
from qafi_core import baseline, features
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
w = WorkspaceClient(profile="qafi")
CATALOG = "qafi_dev"
EXPERIMENT = f"/Users/{w.current_user.me().user_name}/qafi-variant-effect"   # MLflow 实验的名字，图纸里定的
PAPER = ROOT.parent / "QAFI_CODE_NEW/experiment/LOPO_and_ClinVarAUC/outputs/lopo_predictions_qafi2.csv.gz"   # 旧论文的预测，只当参照
mlflow.set_tracking_uri("databricks://qafi")   # MLflow 的记录存在 Databricks，用同一个登录
print("Databricks:", w.config.host)
print("旧论文的预测:", "找到了" if PAPER.exists() else "没找到（第 7 步会跳过）")

---

## 第 1 步 · 特征表是什么

### 在干什么
弄清楚特征表长什么样、里面有哪些特征、为什么要做成一张表。

### 背景
比喻：**体检总表**。07a 说过，特征就是给每个突变开的一张体检报告。07d、07e 各做了几项检查，但结果记在各自的本子上。现在把它们抄到一张总表里：**一行一个突变，一列一项检查，最后一列是诊断结果（标签）**。

这张表叫**特征表**（feature table），在 `gold.fct_variant_features`。表里一共 16 个特征，分 4 组：

| 组 | 特征 | 它在说什么 | 数值怎么读 |
|---|---|---|---|
| **替换本身** | `blosum62` | 这种替换在进化里常不常见 | 负数 = 少见 |
| | `d_hydrophobicity` | 新氨基酸比旧的更亲水还是更疏水 | 新 − 旧 |
| | `d_volume` | 变大还是变小 | 新 − 旧，单位 Å³ |
| | `d_charge` | 电荷变了没有 | 新 − 旧 |
| **语言模型** | `esm_delta` | ESM-2 觉得新氨基酸有多"不像话" | 越负越不像，07e 的零样本分数 |
| | `esm_wt_logprob` | 原来那个氨基酸，模型有多确定就该是它 | 接近 0 = 非它不可 |
| | `esm_entropy` | 这个位置能容下多少种氨基酸 | 0 = 只能一种（保守），3 = 都行 |
| **结构** | `plddt` | AlphaFold 对这个位置有多确定 | 0–100，低的常是无序区 |
| | `rsa` | 埋在里面还是露在外面 | 0 = 埋着，1 = 全露 |
| | `contacts_8`、`contacts_12` | 周围 8 Å、12 Å 内有多少残基 | 多 = 挤 |
| | `hse_up`、`hse_down` | 侧链朝向的那半边、背面那半边各有多少残基 | 半球暴露度 |
| **周围环境** | `neighbour_entropy` | 空间上挨着的那些残基保不保守 | 低 = 邻居都保守，常是功能位点 |
| | `d_volume_buried` | 体积变化 ×（1 − `rsa`） | 在蛋白内部变大变小，比在表面要紧 |
| | `relative_position` | 在链的开头、中间还是末尾 | 0–1 |

**07a 清单里有两项这次没做**，直说：
- **二级结构**：要再装一个算二级结构的工具。先看基线缺不缺它；08 的图神经网络直接看三维结构，不需要这一项
- **Δ接触能量**：要一张外部的势能表（Miyazawa–Jernigan），它还没进 raw。真要用的时候按 03b 的流程把它接进来，不手抄

### 用了什么技术，为什么
**两张表，不是一张**：

```
gold.dim_residue             一个残基一行：这个位置本身的性质（结构、保守性、邻居）
        │                    和"换成什么"无关，所以每个位置只算一次
        ▼
gold.fct_variant_features    一个突变一行 = 残基的性质 + 这次替换的性质 + 标签 + 分在哪一边
```

这样分的原因是 09 号：给全蛋白组打分时，每个位置要算 19 种替换。位置的性质算一次、用 19 次。

**两张小查找表**（dbt 的 **seed**，07a 用过）：`amino_acid`（20 种氨基酸的疏水性、体积、电荷）和 `blosum62`（400 对替换的分数）。它们由 `src/qafi_core/features.py` 里的函数生成，有一个测试专门检查"仓库里的文件和函数生成的一模一样"，所以没人能悄悄改一个数。

**用 dbt 的 SQL 来拼，而不是写 Python 脚本**，三个理由：
1. **训练和预测走同一段 SQL**。`dim_residue` 现在只算有标签的 494 个蛋白；09 号把一个开关（`residue_scope`）拨到 `proteome`，同一段 SQL 就算全部 20,652 个蛋白。07a 说的 training-serving skew（训练时和使用时特征算法不一样）就这样防住
2. **每一列的来历能点出来**。Databricks 的血缘图（Lineage）自动画出这张表是从哪几张表来的
3. **测试跟着表走**。每次建表都自动检查：行数对不对、有没有重复、有没有丢突变

### 公司里什么情况怎么做
- 特征表是 ML 团队和数据团队的**交接面**：数据团队保证表是对的，建模的人只管读这张表。所以它要有文档、有测试、有负责人，和 05 号的 gold 表一个待遇
- 带主键的特征表，在 Databricks 里可以直接登记成"特征存储"（Feature Store）给线上服务用。我们的 API（10 号）读的是预测结果，不用这一步，知道有这回事就行
- **新加一个特征** = 改 SQL、加一行文档、在 `features.py` 的清单里加一个名字。有测试检查清单里的每个名字在表里都有文档

### 面试怎么说
> "The feature table is a dbt model with one row per variant per assay: residue-level features are computed once per position in a dimension table and joined with substitution lookups from seeds. The same SQL computes features for training and for proteome-wide inference, which rules out training-serving skew."

---

## 第 2 步 · 建表

### 在干什么
让 dbt 在 Databricks 上建出这两张表，同时跑它们的检查。

### 背景
05 号学过：`dbt build` = 按依赖顺序建表 + 跑测试。这次只建和特征有关的那一部分，用 `--select` 点名：两张查找表、两个 staging 视图，名字后面的 `+` 表示"**连同它下游的所有东西**"，也就是 `dim_residue`、`fct_variant_features` 和它们的测试。

### 执行

In [ ]:
os.environ.update(lakehouse.dbt_env(w))
run("cd dbt && uv run dbt build --select amino_acid blosum62 stg_residue_structure+ stg_residue_language_model+ --profiles-dir .")

### 结果怎么看
最后一行 `Done. PASS=31 WARN=0 ERROR=0`，31 项里有：

| 类别 | 数量 | 例子 |
|---|---|---|
| 查找表（seed） | 2 | `amino_acid`、`blosum62` |
| 视图 | 2 | `stg_residue_structure`、`stg_residue_language_model` |
| 表 | 2 | `dim_residue`（约 40 秒）、`fct_variant_features`（约 40 秒） |
| 单元测试 | 2 | 拿几行编出来的数据，检查 SQL 算得对不对 |
| 数据测试 | 23 | 不为空、取值合法、每个突变只有一行、每个标签都进了表 |

**单元测试**是这一章新用的。它不看真实数据，而是给 SQL 喂 3 行手编的数据，检查算出来的是不是预期的数。比如"残基 1 的邻居是残基 2 和 3，它们的熵是 2 和 3，所以邻居平均熵应该是 2.5"。SQL 写错了，这里马上红。

看一个熟悉的突变，TP53 的 R175H，它的"体检报告"：

In [ ]:
sql(w, f"""SELECT * EXCEPT (label) FROM {CATALOG}.gold.fct_variant_features
           WHERE uniprot_id = 'P04637' AND uniprot_position = 175 AND alt_aa = 'H' LIMIT 1""").T

### 结果怎么看
- `esm_delta` 约 **−9.1**，`esm_entropy` 约 **0.01**：ESM-2 认为这个位置几乎只能是精氨酸，换成组氨酸非常不像话。和 07e 第 2 步在你电脑上算的是同一个数
- `rsa` 约 **0.03**，`plddt` **96.6**：埋在蛋白内部，AlphaFold 很确定。和 07d 说的一样
- `blosum62` = **0**：单看氨基酸，R 换 H 不算离谱。**所以只靠"替换本身"看不出 R175H 有害，要靠位置的信息**
- `split` = **test**：TP53 是考试蛋白，训练时看不到它。所以上面那一格特意没取 `label` 这一列：考试蛋白的答案，现在不看

### 去网页上看
Databricks → **Catalog** → `qafi_dev` → `gold` → `fct_variant_features`：
- **Overview**：每一列都有说明，是 dbt 从文档里写进去的
- **Lineage** → **See lineage graph**：往左一路展开，能看到这张表来自标签、`dim_residue`、两张查找表，再往左是 silver、bronze

### 面试怎么说
> "The feature models have dbt unit tests on hand-made rows, so the SQL logic is checked independently of the data, and data tests for grain, completeness and value ranges on every build."

---

## 第 3 步 · 检查特征

### 在干什么
训练之前，确认这张表可以放心用。

### 背景
比喻：**交卷前检查**。模型不会告诉你数据有问题，它只会默默学错。上线前看三件事：

| 检查 | 问的是 | 出了问题会怎样 |
|---|---|---|
| **覆盖** | 每个标签都有特征吗？有没有缺失值？ | 被 join 丢掉的突变悄悄不参加训练，没人知道 |
| **颗粒度** | 一个突变是不是只有一行？ | join 把行变多了，同一个突变被数两遍 |
| **泄漏** | 有没有哪个特征是从答案里算出来的？ | 考试成绩虚高，上线后原形毕露 |

前两项 dbt 的测试已经查过了（`assert_every_label_has_features`、`assert_one_feature_row_per_label`）。这里亲眼看一下数：

In [ ]:
missing = " OR ".join(f"{name} IS NULL" for name in features.FEATURES)
sql(w, f"""SELECT split, data_kind, count(DISTINCT uniprot_id) AS proteins, count(DISTINCT dataset_id) AS datasets,
                  count(*) AS variants, sum(CASE WHEN {missing} THEN 1 ELSE 0 END) AS with_missing_feature
           FROM {CATALOG}.gold.fct_variant_features GROUP BY ALL ORDER BY split DESC, data_kind""")

### 结果怎么看
- 六行加起来 **1,069,464** 个突变，和 07c 的标签数一个不差：没有突变在拼表时丢掉
- `with_missing_feature` 全是 **0**：16 个特征没有一个空值
- **训练**（train）：349 个蛋白、445 组实验、84.1 万个突变。**考试**（test）：108 个蛋白，这一章**一眼都不看**
- `data_kind` 是这一章新加的一列，把数据分成三类，后面一直要用：

| `data_kind` | 是什么 | 测的是 |
|---|---|---|
| `domainome` | Domainome，500 组，一组一个结构域 | 蛋白在细胞里的**丰度**，基本反映稳定性 |
| `proteingym` | ProteinGym 里的热稳定性实验，13 组 | **稳定性**（ΔΔG） |
| `mavedb` | MaveDB 里其他的实验，79 组 | **功能**：酶活、结合、细胞生长…… 五花八门 |

**泄漏**这一项机器查不了，要靠人看清单：16 个特征全部来自序列、AlphaFold 结构和查找表，没有一个用到 DMS 分数。`split` 和 `label` 在表里，但不在特征清单里。

### 公司里什么情况怎么做
- 泄漏最常见的来源不是"故意偷看"，而是某个特征在算的时候**悄悄用到了全体数据**。比如"这个位置所有突变的平均分"，听起来像位置的性质，其实就是答案。原则：**特征只能来自预测时也拿得到的东西**
- 特征有缺失值不是错（树模型能处理），但要**知道**缺多少、为什么缺。我们这里是 0，因为 07d、07e 把所有有标签的蛋白都算全了

### 面试怎么说
> "Before training I check three things on the feature table: every label has a row and no feature is missing unexpectedly, the grain is one row per variant per assay, and no feature is derived from the labels: features may only use what is available at prediction time."

---

## 第 4 步 · 基线模型和考法

### 在干什么
定下第一个模型用什么、怎么考、和谁比。

### 背景
比喻：**对照组**。做实验要有对照，建模也一样。深度学习模型（08 号）跑出一个 0.5，这算好还是不好？要有参照才知道。我们放两根横杆：

| 横杆 | 是什么 | 意义 |
|---|---|---|
| **零样本** | 直接拿 `esm_delta` 当预测，什么都不训练 | 最低的杆：训练出来的模型要是连它都不如，就白训练了 |
| **基线** | 16 个特征 + 梯度提升树 | 第二根杆：深度学习要比它高，才值得那份复杂 |

### 用了什么技术，为什么
**梯度提升树**（gradient-boosted trees），用的库叫 **XGBoost**。
- **一棵决策树**就是一串是非题："埋在里面吗？→ 是 → ESM 分数低于 −8 吗？→ 是 → 预测 0.2"
- **提升**（boosting）：种 400 棵小树，**每一棵专门去纠正前面那些树还没答对的部分**，最后把所有树的回答加起来
- 为什么用它：表格数据（一行一个样本、一列一个特征）上，它至今仍是最稳的选择，常常不输神经网络；几乎不用调参；能告诉你每个特征贡献了多少。公司里拿到一张特征表，第一个模型十有八九是它或它的近亲 LightGBM

**考法**，07a 定的规矩，现在落到代码里（`src/qafi_core/baseline.py`）：

1. **分组 5 折交叉验证**：349 个训练蛋白按近亲组分成 5 份，轮流拿 4 份训练、1 份考。每个蛋白被预测时，模型都没见过它和它的近亲
2. **每组实验一票**：有的实验测了 3 万个突变，有的只测了 800 个。不加处理，模型就只会讨好那几个大实验。所以给每个突变一个**权重**，让每组实验的总分量一样
3. **成绩**：每组实验算一个 **Spearman 相关系数**（预测的排序和实验的排序有多一致），再取**中位数**。整体报一个，三类数据各报一个

**怎么算"比它好"**：两个中位数差 0.01，可能只是运气。标准做法是**配对比较**：同一组实验，两个模型各得一个分，看差值。445 组实验就有 445 个差值，再用 **Wilcoxon 符号秩检验**看这些差值是不是明显偏向一边。这和"同一批病人用药前后比"是一个道理。

### 执行
这些规矩都有测试。在你电脑上跑一遍，用的是编出来的小数据，几秒钟：

In [ ]:
run("uv run pytest tests/unit/test_baseline.py tests/unit/test_features.py -q")

### 结果怎么看
`12 passed`。其中几个测试保证的事：
- 交叉验证里，**近亲永远在同一折**
- **每组实验的总权重相等**
- 限定"只用某几类数据训练"时，其他数据照样被预测（第 6 步的对照实验要用）
- 仓库里的查找表和代码生成的**一模一样**；SQL 和 Python 读 ESM 那 20 个概率的**顺序一致**

### 公司里什么情况怎么做
- **先基线，后复杂模型**。基线一天就能做完，它告诉你这个问题大概能做到多好、数据有没有毛病。跳过基线直接上深度学习，出了问题分不清是模型的错还是数据的错
- **科学代码和平台代码分开**。`baseline.py` 里没有一行 MLflow 或 Databricks 的代码，所以能在你电脑上测；Databricks 上的那个任务只负责喂数据、记结果

### 面试怎么说
> "The baseline is XGBoost on 16 interpretable features, evaluated by grouped 5-fold cross-validation with homologous proteins kept in one fold and each assay weighted equally. Models are compared assay by assay with a paired Wilcoxon test, not by eyeballing two medians."

---

## 第 5 步 · MLflow：把每次训练记下来

### 在干什么
在 Databricks 上训练，并且让每一次训练都自动留下完整的记录。

### 背景
比喻：**实验记录本**。你做湿实验时，每次都要记：用的哪批试剂、什么方案、结果多少、样品放在哪个冰箱。半年后有人问"图 3 那个数怎么来的"，翻本子就知道。

训练模型也是做实验，而且做得更多：换一组特征、少用一类数据、调一个参数，都是一次新实验。不记的下场，旧 QAFI 代码里就有：结果散在几十个 CSV 里，文件名叫 `final_v2_new`；论文里的模型一个都没存下来。

**MLflow** 就是机器学习的实验记录本，开源，Databricks 自带。四个词：

| MLflow 的词 | 是什么 | 对应实验记录本 |
|---|---|---|
| **Experiment**（实验） | 一个课题下所有训练的集合 | 一个本子 |
| **Run**（一次运行） | 一次训练 | 本子里的一页 |
| **Parameters / Metrics** | 这次的设置 / 这次的成绩 | 方案 / 结果 |
| **Artifacts / Model** | 这次产出的文件 / 训练好的模型 | 样品 |

### 用了什么技术，为什么
**每一页都记三样"来历"**，有了它们，任何一个数字都能重做出来：

| 记什么 | 我们记的是 | 回答的问题 |
|---|---|---|
| **代码版本** | 部署时的 git 提交号 | 用的哪一版代码 |
| **数据版本** | 特征表的 Delta 版本号（04 号讲过，Delta 表每次改动都有版本） | 用的哪一版数据 |
| **参数** | 用了哪些特征、哪几类数据、树的设置 | 怎么设的 |

**图纸里加了三样东西**（`databricks/resources/lakehouse.yml`）：
- 一个 MLflow 实验 `qafi-variant-effect`：以后所有模型（基线、08 号的深度学习）都记在这一个本子里，才能并排比
- 一层新楼 `ml`：放模型的产出。`ml.cv_scores` 是每次运行、每组实验的成绩；`ml.cv_predictions` 是基线对每个训练突变的预测
- 一个新流水线 `qafi baseline model`，两步：`features`（就是第 2 步的 dbt）→ `train_baseline`（训练）

**一次运行记 8 页**：零样本这根杆、基线，再加 6 个"拿掉一样东西"的对照（第 6 步看）。这些对照在代码里有名有姓（`baseline.RUNS`），不是临时改参数跑出来的。

### 执行
部署图纸，运行流水线。**约 13 分钟**，大部分时间是 Databricks 在准备机器、装 XGBoost、读数据；训练本身很快，在你的 Mac 上一次交叉验证只要十几秒。

In [ ]:
run("cd databricks && databricks bundle deploy")
run("cd databricks && databricks bundle run baseline")
lakehouse.latest_run(w, "qafi baseline model")

### 结果怎么看
两个任务 `features`、`train_baseline` 都是 `SUCCESS`。

### 去网页上看
这一步的重点在网页上，花两分钟点一遍：

1. Databricks 左边栏 → **Experiments** → `qafi-variant-effect`
2. 列表里每一行是一次运行（run）。我写这一章时已经跑过一遍，你再跑一遍，**每个名字就有两行**：MLflow 从不覆盖，每跑一次多一页
3. 勾选 `baseline` 和 `esm2_zero_shot` → **Compare**：两页并排，参数哪里不同、指标差多少，一目了然
4. 点进 `baseline`：
   - **Overview**：参数（`feature_table_version`、`n_estimators`……）、指标、标签里的 `code_version`
   - **Artifacts**：`dataset_scores.json`（445 组实验各自的分数）、`feature_importance.json`
   - **Logged models** 一栏 → `model`：训练好的模型，带着它需要哪 16 列输入的说明。09 号会从这里把它取出来用

Databricks → **Jobs & Pipelines** → `qafi baseline model`：两个方框。

### 公司里什么情况怎么做
- **什么都记，自动记**。靠人手抄进 Excel 的记录，三个月后一定不全。把记录写进训练代码，跑一次记一次
- **模型不是文件，是有来历的东西**。存在 MLflow 里的模型带着：谁训练的、用什么数据、成绩多少、输入长什么样。09 号要把它"注册"成正式版本，前提就是这些都在
- **小实验攒在一起跑**。这次 8 个运行放在一个任务里顺序跑，因为机器启动要好几分钟，训练只要十几秒

### 面试怎么说
> "Every training run is logged to MLflow with the three things needed to reproduce it: the git commit, the Delta version of the feature table and the parameters; plus metrics, per-assay scores and the model with its input signature. Ablations are named runs in code, so the comparison table is reproducible too."

---

## 第 6 步 · 看结果：过线了吗，每样东西贡献多少

### 在干什么
把 MLflow 里的记录读回来，回答三个问题：基线比零样本强吗？每类特征贡献多少？每类数据贡献多少？

### 背景
比喻：**敲除实验**。想知道一个基因有什么用，就把它敲掉，看少了什么。建模里叫**消融实验**（ablation）：拿掉一组特征或一类数据，重新训练，看成绩掉多少。掉得多，说明它重要；不掉，说明可有可无。

### 执行
每个名字取最新的一次运行，按整体成绩排序：

In [ ]:
mlflow.set_experiment(EXPERIMENT)
runs = mlflow.search_runs(order_by=["start_time DESC"]).drop_duplicates("tags.mlflow.runName").set_index("tags.mlflow.runName")
kinds = {"metrics.spearman_median": "all", "metrics.spearman_median_domainome": "domainome",
         "metrics.spearman_median_proteingym": "proteingym", "metrics.spearman_median_mavedb": "mavedb"}
runs[list(kinds)].rename(columns=kinds).rename_axis("run").sort_values("all", ascending=False).round(3)

### 结果怎么看
（输出开头如果有一行橙色的英文，是 Databricks 给 MLflow 另一个功能做的提示，和我们无关。）

数字是每组实验的 Spearman 的**中位数**，越高越好（不同机器上可能差 0.005 左右）：

| 运行 | 是什么 | 全部 | domainome | proteingym | mavedb |
|---|---|---|---|---|---|
| `baseline` | **基线：16 个特征，全部数据** | **0.550** | 0.572 | 0.583 | 0.409 |
| `esm2_zero_shot` | 横杆：不训练，直接用 ESM 分数 | 0.455 | 0.461 | 0.468 | 0.398 |
| `only_language_model` | 只用语言模型那 3 个特征 | 0.443 | 0.447 | 0.436 | 0.378 |
| `only_structure` | 只用结构那 6 个特征 | 0.380 | 0.383 | 0.496 | 0.269 |
| `only_substitution` | 只用替换本身那 4 个特征 | 0.343 | 0.366 | 0.319 | 0.239 |
| `without_neighbourhood` | 拿掉"周围环境"那一组 | 0.558 | 0.576 | 0.542 | 0.415 |
| `without_domainome` | 训练时不用 Domainome | 0.470 | 0.482 | 0.589 | 0.394 |
| `only_domainome` | 训练时只用 Domainome | 0.548 | 0.580 | 0.562 | 0.402 |

**每类特征贡献多少**
- 单独用哪一组都不行：最好的是语言模型 0.443，结构 0.380，替换本身 0.343。**三组合起来 0.550**，比任何一组单独用高 0.1 以上。它们说的是不同的事，能互补
- 结构特征单独用，在稳定性数据（proteingym）上有 0.496，在功能数据（mavedb）上只有 0.269。**"埋不埋"决定稳定性，决定不了功能**
- "周围环境"那一组，拿掉以后整体反而是 0.558。差 0.008 在噪音范围内，结论是：**这一组没有证明自己有用**。先留着，因为"邻居"正是 08 号图神经网络要好好做的事，到时候再看

**每类数据贡献多少**（这回答了 07b 留下的问题：Domainome 该不该进训练集）
- 不用 Domainome，Domainome 那一栏从 0.572 掉到 0.482：稳定性这门课，没有它就学不会
- 不用 Domainome，功能数据那一栏是 0.394；用了是 0.409。**加进稳定性数据，功能预测没有变差，还略好一点**。结论：**留着**
- 只用 Domainome 训练，功能数据也有 0.402。也就是说，功能数据自己那 29 个蛋白，对功能预测几乎没帮上忙

最要紧的一个问题：基线到底比零样本强不强？看配对比较，一类数据一行：

In [ ]:
versus = runs.loc["baseline"].filter(like="vs_zero_shot").rename(lambda name: name.replace("metrics.vs_zero_shot_", ""))
pd.DataFrame({what: versus.filter(like=what).rename(lambda name: name.replace(what + "_", ""))
              for what in ["better", "difference", "p_value"]}).astype(float).round(3)

### 结果怎么看
三列的意思：`better` = 基线比零样本高的实验占多少；`difference` = 平均高多少；`p_value` = 如果两者其实一样，碰巧出现这种差距的可能性（小于 0.05 算有真差别）。

| 数据 | 基线更好的实验占 | 平均高多少 | p 值 | 结论 |
|---|---|---|---|---|
| domainome | 88% | +0.092 | < 0.001 | **基线明显更好** |
| proteingym | 100%（10 组全赢） | +0.119 | 0.002 | **基线明显更好** |
| mavedb | 41% | −0.003 | 0.42 | **没有差别** |

**这是这一章最重要的发现，要说实话：**

> 基线在**稳定性**上比零样本高出一大截；在**功能**实验上，和什么都不训练的 ESM-2 打平。

为什么会这样：
- **稳定性有通用的规律**：埋在里面的位置换成大的、带电的氨基酸，蛋白就不稳。这条规律从 313 个蛋白里学得到，换到新蛋白照样成立。我们的结构特征正好对口
- **功能没有通用的规律**：这个酶靠这几个残基催化，那个受体靠那一片结合。"哪里是功能位点"每个蛋白都不一样，而训练集里功能数据只有 29 个蛋白，测法又各不相同。16 个简单特征学不出"功能位点在哪"
- 这和 07b 当时的判断一致：**缺的是蛋白的个数，不是突变的个数**

我试过的、没有用的办法（都记在这里，省得以后再试）：换权重的算法、调树的深度和棵数、告诉模型"这组实验测的是稳定性还是功能"。功能那一栏始终在 0.39–0.42 之间。

最后看基线最看重哪些特征：

In [ ]:
mlflow.load_table("feature_importance.json", run_ids=[runs.loc["baseline", "run_id"]])

### 结果怎么看
`share` 是每个特征对模型的贡献占比（树每次用它来分叉时，误差减少了多少）：
- **`esm_delta` 和 `rsa` 各占约 23%**，加起来快一半：一个说"进化允不允许"，一个说"埋不埋"
- 接着是 `neighbour_entropy`、`plddt`、`hse_up`，各 7–9%
- 替换本身那四个（`blosum62`、`d_volume`……）都很低：ESM 的分数已经把"这种替换常不常见"包含进去了

注意：重要性说的是"模型用了它多少"，不是"拿掉它会掉多少"。上面的消融实验才回答后一个问题，两个要一起看。

### 公司里什么情况怎么做
- **没用的结果也要记下来**。"这个办法试过，没用"和"这个办法有用"一样值钱，它省掉后面的人三天时间
- **成绩要分开报**。只报一个总分 0.550，看起来基线大获全胜；分开报，才看见功能那一栏根本没动。上线后挨骂的往往就是被总分盖住的那一块
- **基线定了就不再改**。它是横杆，横杆不能跟着跳高的人一起动

### 面试怎么说
> "The baseline beats zero-shot ESM-2 clearly on stability-type assays, plus 0.09 to 0.12 Spearman with p below 0.01, but ties it on functional assays. Ablations showed each feature family is weak alone and that adding stability data did not hurt function. The bottleneck is the number of proteins with functional data, not the number of variants, and that sets the target for the deep-learning models."

---

## 第 7 步 · 和旧论文同题对照

### 在干什么
新基线和旧论文的 QAFI，在同一批突变上比一比。

### 背景
比喻：**同一张卷子，两个人答**。不同的数据、不同的标签上得出的两个数没法比。要比，就得：
- **同一批突变**：两边都有的蛋白、都有的突变
- **同一个标准答案**：都用旧论文自己的标签打分
- **同一种考法**：旧论文做过"留一蛋白"（每次留一个蛋白不训练，预测它）；我们的交叉验证也是没见过这个蛋白。两边都是"第一次见"

旧论文的预测在 `QAFI_CODE_NEW` 里，只在你电脑上，**不进平台**，只当参照。我们这边的预测从 `ml.cv_predictions` 取。顺便把 ESM-2 零样本也放进来比。

### 执行

In [ ]:
if PAPER.exists():
    paper = pd.read_csv(PAPER)
    proteins = ", ".join(repr(u) for u in paper.uniprot.unique())
    ours = sql(w, f"""
        SELECT p.uniprot_id, p.uniprot_position, p.ref_aa, p.alt_aa, first(p.predicted) AS predicted, first(f.esm_delta) AS esm_delta
        FROM {CATALOG}.ml.cv_predictions p
        JOIN {CATALOG}.gold.fct_variant_features f USING (dataset_id, uniprot_id, uniprot_position, alt_aa)
        WHERE p.run_name = 'baseline' AND p.uniprot_id IN ({proteins}) GROUP BY ALL""").astype({"uniprot_position": int, "predicted": float, "esm_delta": float})
    versus_paper = baseline.compare_with_paper(ours, paper)
    display(versus_paper)
    print("中位数:", versus_paper[["paper_qafi2", "new_baseline", "esm2_zero_shot"]].median().round(3).to_dict(),
          "|", len(versus_paper), "个蛋白,", f"{versus_paper.variants.sum():,}", "个突变")

### 结果怎么看
旧论文的 60 个蛋白里，有 14 个在我们的训练集这一边（其余的不是人类蛋白、没拿到标签，或者是我们的考试蛋白，不能看），共 8.9 万个突变。每个蛋白一个 Spearman，取中位数：

| | Spearman 中位数 |
|---|---|
| 旧论文 QAFI2（留一蛋白） | 0.461 |
| 新基线（交叉验证） | 0.454 |
| ESM-2 零样本 | 0.502 |

- **新基线和旧论文打平**。14 个蛋白里各有胜负（PRKN、VKOR1、HMDH 新的好；CBS、HXK4、UBE2I 旧的好），配对检验分不出高低
- 这个"打平"是有分量的：旧 QAFI 要 PSI-BLAST 搜一百多 GB 的序列库、要 Arpeggio、要十几张手工统计表，全蛋白组算不了；新基线**每一个特征都能从 raw 重算，对 20,652 个蛋白都算得出来**。同样的成绩，换来了可复现和可扩展
- **ESM-2 零样本在这 14 个蛋白上比两个训练出来的模型都高一点**。这些正是功能实验，和第 6 步的结论一致。我另外在旧论文全部 30 个可比的人类蛋白上核对过：QAFI2 0.484，ESM-2 零样本 0.504，统计上同样分不出高低（p = 0.73）

**要诚实说明的两点**
- 旧论文的"留一蛋白"没有把近亲一起留出去，我们的交叉验证留了。所以这个比法对旧论文略微有利
- 只有 14 个蛋白，结论只能说"打平"，不能说谁更好。真正的终审在 09 号：新模型和旧预测在**考试蛋白**上、同一批突变上比

### 公司里什么情况怎么做
- 换掉一个旧系统之前，一定要有一次**同题对照**。"新系统在新数据上 0.55，旧系统在旧数据上 0.52"这种话没有任何意义
- 打平也可以是换的理由，只要新的在别的方面更好：能复现、能扩展、便宜、好维护。要把这笔账说清楚，而不是硬说成绩更高

### 面试怎么说
> "Against the published model on identical variants and labels, both predicted without seeing the protein, the new baseline ties: median Spearman 0.45 versus 0.46 on 14 proteins. The gain is elsewhere: every feature is reproducible from raw data and available for all 20,000 human proteins, which the old pipeline could not do."

---

## 提交代码

In [ ]:
run("git add -A")
run("uv run pre-commit run", check=False)   # 第一次会自动整理 notebook 的格式
run("git add -A")
run('(git diff --cached --quiet || git commit -q -m "feat: 07f feature table, baseline model and MLflow tracking") && git push -q')

推送后去 GitHub 的 Actions 页面看一眼，应该全绿。

---

## 本章小结

| 步骤 | 做了什么 | 记住什么 |
|---|---|---|
| 1 特征表 | 16 个特征、4 组；残基一张表，突变一张表 | 训练和预测走同一段 SQL |
| 2 建表 | dbt 建表，31 项检查 | SQL 也能写单元测试 |
| 3 检查 | 107 万个突变一个不少，没有缺失 | 覆盖、颗粒度、泄漏 |
| 4 基线 | XGBoost，分组 5 折，每组实验一票 | 先基线，后复杂模型；配对比较 |
| 5 MLflow | 8 次运行全部记录在案 | 代码版本 + 数据版本 + 参数 |
| 6 结果 | 整体 0.550，零样本 0.455 | 稳定性大赢，功能打平；成绩要分开报 |
| 7 旧论文 | 同题对照，打平 | 换来的是可复现、可扩展 |

**一句话**：三类特征拼成一张有测试的表，训练出第一个模型，每次训练都记录在案。基线把稳定性学会了，功能还没有，这就是 08 号要解决的问题。

**07 号整个做完了。** 从 07a 到 07f：定规矩 → 找数据 → 定标签 → 结构特征 → 语言模型特征 → 特征表和基线。

## 面试题

1. **Why build a baseline before a deep-learning model?**
   > "It takes a day, exposes data problems early and gives the bar a complex model has to clear. Ours showed that stability is learnable from simple features while function is not, which told us where deep learning has to add value."

2. **What do you log for each training run, and why?**
   > "The git commit, the version of the feature table and the parameters, so the run can be reproduced; metrics overall and per data type, so a good average cannot hide a weak segment; and the model with its input signature, so it can be registered and served later."

3. **How do you decide whether model A is better than model B?**
   > "On the same held-out assays, paired: the per-assay difference with a Wilcoxon signed-rank test. A difference of 0.01 between two medians is often noise."

4. **What is an ablation, and what did yours show?**
   > "Retraining with one component removed. Each feature family alone was weak and the combination was 0.1 higher; removing the abundance data broke stability prediction but did not help function, so we kept it."

5. **What is training-serving skew and how do you prevent it?**
   > "Features computed differently for training and for inference. Our residue features come from one dbt model with a scope switch, so the proteome-wide run uses the very same SQL."

**下一步 08a · PyTorch 入门**：把 07e 存好的 ESM 向量（每个残基 1,280 个数字）喂给一个小的神经网络。目标很明确：**功能数据那一栏，超过 0.40**。